In [ ]:
# 🛠️ Setup: install deps + fetch the shared helper on Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -q transformer_lens datasets transformers plotly
    # transformer_lens pulls a newer torch; Colab's torchaudio was built against the
    # old torch and fails to load (undefined symbol: torch_library_impl). transformers
    # imports it lazily and crashes, though nothing here uses audio. Remove it.
    !pip uninstall -y -q torchaudio
    !wget -q https://raw.githubusercontent.com/barmag/fanous-llm-lens/main/notebooks/education/tiny.py
import tiny
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

torch.manual_seed(42)
print("device:", tiny.device())

<div dir="rtl">

# المرحلة ٢د: الـ MLP — حساب لكل توكن لوحده

المراحل ٢أ–٢ج كانت **انتباه بس** (attention-only) — عن قصد، عشان ده اللي ورقة ٢٠٢١ بتحلّله بوضوح. بس كتلة المحوّل الحقيقية فيها حاجة تانية: **الـ MLP**.

الفرق الجوهري:
- **الانتباه بيحرّك معلومة بين المواضع** (توكن بيبص لتوكن).
- **الـ MLP بيحسب لكل توكن لوحده** — بياخد متجه الموضع ويحوّله، **من غير ما يبص لأي موضع تاني**.

النهارده بنضيف الـ MLP عشان نكمّل الكتلة، ونثبت بالأرقام إنه فعلًا **مبيحركش معلومة بين المواضع**.

**ملاحظة صادقة:** هنا بتقف القصة النظيفة بتاعة الدوائر في ورقة ٢٠٢١. فهم *إيه* اللي الـ MLP بيخزّنه (features) أصعب بكتير — وده **الجسر** للمرحلة الجاية (المعالم/الـ SAEs).

</div>

<div dir="ltr">

# Stage 2d: The MLP — per-token computation

Rungs 2a–2c were **attention-only** — deliberately, because that's what the 2021 framework analyses cleanly. But a real transformer block has one more part: the **MLP**.

The essential difference:
- **attention moves information between positions** (a token looks at other tokens).
- **the MLP computes on each token alone** — it transforms a position's vector **without looking at any other position**.

Today we add the MLP to complete the block, and prove numerically that it **moves no information between positions**.

**Honest note:** this is where the 2021 framework's clean circuit story stops. Understanding *what* the MLP stores (features) is much harder — that's the **bridge** to the next stage (features / SAEs).

</div>

<div dir="rtl">

## ١. البيانات والمُجزّئ · Data & tokenizer

نفس إعداد ٢أ/٢ب/٢ج: نص عربي + مُجزّئ mGPT مع تصغير القاموس.

</div>

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import re

MAX_CHARS = globals().get("MAX_CHARS", 500_000)

msa_stream = load_dataset("wikimedia/wikipedia", "20231101.ar", split="train", streaming=True)
eg_tweets = load_dataset("amgadhasan/arabic_tweets_dialects", split="train").filter(
    lambda x: x["dialect"] == "EG")

def clean(t):
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[a-zA-Z0-9_@]+", "", t)
    return re.sub(r"[^\s\u0621-\u064A]", "", t)

def collect(rows, key, n):
    out = ""
    for r in rows:
        out += clean(r[key]) + " "
        if len(out) >= n:
            break
    return out[:n]

mgpt = AutoTokenizer.from_pretrained("ai-forever/mGPT")
encode, corpus_ids, id_to_str, VOCAB = tiny.make_compact_encoder(
    mgpt, [collect(msa_stream, "text", MAX_CHARS), collect(eg_tweets, "text", MAX_CHARS)])
all_ids = corpus_ids[0] + corpus_ids[1]
print(f"compact vocab: {VOCAB} subwords  |  tokens: {len(all_ids)}")

<div dir="rtl">

## ٢. نبني الكتلة الكاملة وندرّبها · Build the full block & train

نفس موديل ٢ج (طبقتين، رأسين) بس مع **تشغيل الـ MLP** (`attn_only=False`). كده عندنا كتلة محوّل كاملة: انتباه + MLP.

</div>

In [ ]:
N_CTX = globals().get("N_CTX", 64)
N_EPOCHS = globals().get("N_EPOCHS", 200)

batches = tiny.make_natural_batches(all_ids, n_ctx=N_CTX)
print("batches:", tuple(batches.shape))  # ← (N, n_ctx)

model = tiny.make_tiny_model(n_layers=2, n_heads=2, d_vocab=VOCAB, n_ctx=N_CTX, attn_only=False)
print("attn_only:", model.cfg.attn_only, "| has MLP:", hasattr(model.blocks[0], "mlp"))
losses = tiny.train(model, batches, n_epochs=N_EPOCHS, log_every=20)
print("loss:", round(losses[0], 3), "->", round(losses[-1], 3))  # ← scalar

<div dir="rtl">

## ٣. الـ MLP بيشتغل على كل موضع لوحده · The MLP is per-position

هنثبت ده بالأرقام: ناخد المتجه الداخل للـ MLP عند **موضع واحد بس**، نمرّره في الـ MLP لوحده، ونقارنه باللي طلع في السياق الكامل. لو الفرق **صفر**، يبقى الـ MLP مبيستعملش أي موضع تاني — حساب محلي بحت. وبالعكس، نغيّر توكن في الأول ونشوف **الانتباه** بيتغيّر عند آخر موضع (لأنه بيحرّك معلومة).

</div>

In [ ]:
prompt = "القطة بتاكل السمك"
ids = torch.tensor([encode(prompt)]).to(tiny.device())
_, cache = model.run_with_cache(ids)

# (a) MLP is per-position: applying it to ONE position alone gives the same vector
#     as applying it to the whole sequence -> it never looks at other positions.
mlp = model.blocks[0].mlp
mlp_in = cache["normalized", 0, "ln2"]     # input to the layer-0 MLP  ← (1, seq, d_model)
pos = ids.shape[1] - 1
full = mlp(mlp_in)                          # MLP over every position at once
alone = mlp(mlp_in[:, pos:pos + 1, :])     # MLP over just the last position
print("MLP at the last position: whole-sequence vs in-isolation — max diff:",
      float((full[0, pos] - alone[0, 0]).abs().max()))   # ← exactly 0  => no token mixing

# (b) attention DOES mix: change the FIRST token, the last position's attn output shifts.
ids2 = ids.clone(); ids2[0, 0] = (int(ids2[0, 0]) + 1) % VOCAB
_, cache2 = model.run_with_cache(ids2)
attn_shift = (cache["attn_out", 0][0, -1] - cache2["attn_out", 0][0, -1]).abs().max()
print("changing the first token shifts attention at the last position by:",
      float(attn_shift))   # ← > 0  => attention moves information between positions

<div dir="rtl">

## ٤. الكتلة الكاملة على شجرة التوقعات · The full block on the prediction tree

نعيد **شجرة التوقعات** على نفس السياقين بتوع كل السُلّم — دلوقتي بكتلة محوّل كاملة (انتباه + MLP). قارن بـ ٢أ/٢ب/٢ج: ده الموديل الأقرب لمحوّل حقيقي اللي بنيناه. (أحمر = `[UNK]` أو احتمال `< 0.10`.)

</div>

In [ ]:
LOW_PROB = 0.10

def next_topk(context_ids, k):
    ids = torch.tensor([context_ids]).to(tiny.device())
    probs = torch.softmax(model(ids)[0, -1], dim=-1)
    vals, idx = torch.topk(probs, k)
    return [(int(i), float(v)) for v, i in zip(vals, idx)]

def build_prediction_tree(context_text, max_depth=2, top_k=2):
    edges, nodes = [], {"root": (context_text, True)}
    frontier, n = [("root", encode(context_text), 0)], 0
    while frontier:
        pkey, ctx, depth = frontier.pop(0)
        if depth >= max_depth:
            continue
        for tid, p in next_topk(ctx, top_k):
            n += 1
            label = id_to_str.get(tid, "[UNK]")
            sensible = (label != "[UNK]") and (p >= LOW_PROB)
            ckey = f"{depth}:{n}:{label}"
            nodes[ckey] = (label, sensible)
            edges.append((pkey, ckey, p, depth, sensible))
            frontier.append((ckey, ctx + [tid], depth + 1))
    return edges, nodes

def tree_layout(edges):
    depth_of = {"root": 0}
    for _pk, ck, _p, d, _s in edges:
        depth_of[ck] = d + 1
    levels = {}
    for k, d in depth_of.items():
        levels.setdefault(d, []).append(k)
    return {k: (d, (len(ks) - 1) / 2 - i)
            for d, ks in levels.items() for i, k in enumerate(ks)}

def add_tree(fig, col, edges, nodes):
    sfx = "" if col == 1 else str(col)
    xref, yref = f"x{sfx}", f"y{sfx}"
    pos = tree_layout(edges)
    for pk, ck, p, _d, s in edges:
        (x0, y0), (x1, y1) = pos[pk], pos[ck]
        color = "#c0392b" if not s else "#2b6cb0"
        fig.add_annotation(x=x1, y=y1, ax=x0, ay=y0, xref=xref, yref=yref, axref=xref,
                           ayref=yref, showarrow=True, arrowhead=3, arrowwidth=1.5 + p * 7.5,
                           arrowcolor=color, standoff=18, startstandoff=24, text="")
        fig.add_annotation(x=(x0 + x1) / 2, y=(y0 + y1) / 2, xref=xref, yref=yref,
                           showarrow=False, text=f"{p:.2f}", font=dict(size=10, color="#333"),
                           bgcolor="rgba(255,255,255,0.75)")
    keys = list(pos.keys())
    fig.add_trace(go.Scatter(
        x=[pos[k][0] for k in keys], y=[pos[k][1] for k in keys], mode="markers+text",
        text=[nodes[k][0] for k in keys], textposition="middle center",
        textfont=dict(size=13, color="#111"),
        marker=dict(color=["#f6c350" if k == "root" else ("#f6c0bb" if not nodes[k][1] else "#d9d9d9")
                           for k in keys],
                    size=[40 if k == "root" else 30 for k in keys], line=dict(width=1.5, color="#333")),
        hoverinfo="text", showlegend=False), row=1, col=col)
    fig.update_xaxes(visible=False, row=1, col=col)
    fig.update_yaxes(visible=False, row=1, col=col)

CONTEXTS = ["القطة بتاكل السمك", "الولد بياكل السمك"]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=[f"السياق: {c}" for c in CONTEXTS])
for i, c in enumerate(CONTEXTS):
    e, n = build_prediction_tree(c, max_depth=2, top_k=2)
    add_tree(fig, i + 1, e, n)
fig.update_layout(height=460, margin=dict(l=20, r=20, t=80, b=20),
                  title_text="كتلة كاملة (انتباه + MLP) · full block — prediction trees")
fig.show()

<div dir="rtl">

## ٥. الجسر للمعالم · The bridge to features

ضفنا الـ MLP وأثبتنا إنه حساب محلي لكل توكن. بس **إيه** اللي بيحسبه؟ المتجهات اللي جوّاه بتخزّن **معالم** (features) متشابكة — مش سهل نقراها زي ما قرينا رأس الانتباه. ده موضوع المرحلة الجاية: **المعالم والـ SAEs** (sparse autoencoders) اللي بتحاول تفك تشابك المعالم دي.

</div>

<div dir="rtl">

## ٦. خلاصة السُلّم كله · Recap of the whole ladder

بنينا محوّل **حتة حتة** على بيانات عربي:

١. **تضمينات** (١ج) — متجه لكل subword، أعمى للسياق.
٢. **+ كتلة انتباه** (٢أ) — التوكن بقى يبص للي قبله؛ السياق بيغيّر التوقع.
٣. **+ كذا رأس** (٢ب) — كذا علاقة بالتوازي.
٤. **+ طبقة تانية** (٢ج) — التركيب → **رأس الاستقراء**.
٥. **+ MLP** (٢د) — حساب محلي لكل توكن؛ الكتلة اكتملت.

من هنا بيكمل الطريق للمعالم والـ SAEs.

</div>

<div dir="ltr">

## 6. Recap of the whole ladder

We built a transformer **one piece at a time** on Arabic data:

1. **embeddings** (1c) — a vector per subword, context-blind.
2. **+ attention block** (2a) — a token can look at earlier tokens; context changes the prediction.
3. **+ multiple heads** (2b) — several relations in parallel.
4. **+ a second layer** (2c) — composition → the **induction head**.
5. **+ MLP** (2d) — per-token computation; the block is complete.

From here the path continues to features and SAEs.

</div>